## Test cases run


In [5]:
import numpy as np
import serial
import time
import csv
from scipy.fft import fft, fftfreq
from scipy.optimize import minimize

# =====================================================
# SERIAL
# =====================================================
ser = serial.Serial('COM4',115200,timeout=1)
time.sleep(2)

# =====================================================
# ROBOT PARAMETERS
# =====================================================
DOF = 4
T = 80
dt = 0.04

L1 = 0.05
L2 = 0.11
L3 = 0.11

# =====================================================
# PREDICTIVE MODEL STORAGE
# =====================================================
last_fft = 1.0
last_acc_cost = 1.0

# =====================================================
# FORWARD KINEMATICS
# =====================================================
def forward_kinematics(q):

    q1,q2,q3,q4 = q

    r = L2*np.cos(q2) + L3*np.cos(q2+q3)
    x = np.cos(q1)*r
    y = np.sin(q1)*r
    z = L1 + L2*np.sin(q2) + L3*np.sin(q2+q3)

    return np.array([x,y,z])

# =====================================================
# READ CURRENT ANGLES
# =====================================================
def read_current_angles():

    while True:
        line = ser.readline().decode(errors='ignore').strip()

        if line.startswith("ANG:"):
            try:
                angles = line.replace("ANG:","")
                base, shoulder, elbow, gripper = map(float, angles.split(","))
                return np.radians([base, shoulder, elbow, gripper])
            except:
                continue

# =====================================================
# JACOBIAN
# =====================================================
def jacobian(q):

    q1,q2,q3,_ = q
    r = L2*np.cos(q2) + L3*np.cos(q2+q3)

    J = np.zeros((3,3))

    J[0,0] = -np.sin(q1)*r
    J[1,0] =  np.cos(q1)*r

    J[0,1] = np.cos(q1)*(-L2*np.sin(q2)-L3*np.sin(q2+q3))
    J[1,1] = np.sin(q1)*(-L2*np.sin(q2)-L3*np.sin(q2+q3))
    J[2,1] = L2*np.cos(q2)+L3*np.cos(q2+q3)

    J[0,2] = np.cos(q1)*(-L3*np.sin(q2+q3))
    J[1,2] = np.sin(q1)*(-L3*np.sin(q2+q3))
    J[2,2] = L3*np.cos(q2+q3)

    return J

# =====================================================
# IK
# =====================================================
def inverse_kinematics_NR(target,q_init):

    q = q_init.copy()

    for _ in range(50):

        error = target - forward_kinematics(q)

        if np.linalg.norm(error) < 1e-4:
            break

        dq = np.linalg.pinv(jacobian(q)) @ error
        q[:3] += dq

    q[3] = 0
    return q

# =====================================================
# TRAJECTORIES
# =====================================================
def base_traj(q0,qf):
    t = np.linspace(0,1,T)
    return q0 + np.outer(t,(qf-q0))

def fourier_traj(q0,qf,c):
    t = np.linspace(0,1,T)
    q = base_traj(q0,qf)
    for i in range(DOF):
        q[:,i] += 0.25*c[i]*np.sin(2*np.pi*3*t)
    return q

# =====================================================
# SERVO EXECUTION
# =====================================================
def clamp_servo(rad):

    deg = np.degrees(rad)

    deg[0] = np.clip(deg[0],0,180)
    deg[1] = np.clip(deg[1],0,180)
    deg[2] = np.clip(deg[2],90,180)
    deg[3] = np.clip(deg[3],0,100)

    return deg

def run_traj(traj):

    acc_data = []

    for i in range(T):

        deg = clamp_servo(traj[i])
        deg[3] = 100

        packet = " ".join(str(int(a)) for a in deg) + "\n"
        ser.write(packet.encode())

        line = ser.readline().decode(errors='ignore').strip()

        if line:
            try:
                ax,ay,az = map(float,line.split(","))
                acc_data.append([ax,ay,az])
            except:
                pass

        time.sleep(dt)

    deg = clamp_servo(traj[-1])

    deg[3] = 0
    ser.write((" ".join(str(int(a)) for a in deg) + "\n").encode())
    time.sleep(1)

    deg[3] = 0
    ser.write((" ".join(str(int(a)) for a in deg) + "\n").encode())

    return np.array(acc_data)

# =====================================================
# FFT ENERGY
# =====================================================
def fft_energy(acc):

    if len(acc)<10:
        return 1e9

    acc -= np.mean(acc,axis=0)
    N = len(acc)
    freqs = fftfreq(N,dt)

    fft_total = (
        np.abs(fft(acc[:,0]))**2 +
        np.abs(fft(acc[:,1]))**2 +
        np.abs(fft(acc[:,2]))**2
    )/N

    pos = freqs>0
    freqs = freqs[pos]
    fft_total = fft_total[pos]

    band = (freqs>=4)&(freqs<=7)

    return np.sum(fft_total[band])

# =====================================================
# EXPERIMENT LOOP
# =====================================================

q_A = read_current_angles()
q_start_home = q_A.copy()

pos_B = np.array([0.05, 0.15, 0.20])
pos_C = np.array([-0.10, 0.10, 0.05])
pos_D = np.array([0.15, -0.10, 0.18])
pos_E = np.array([-0.18, -0.15, 0.12])
pos_F = np.array([0.18, 0.18, 0.10])

positions = [pos_B,pos_C,pos_D,pos_E,pos_F]

normal_results = []
opt_results = []
return_results = []

q_prev = q_A.copy()
joint_points = []

for p in positions:
    q_sol = inverse_kinematics_NR(p,q_prev)
    joint_points.append(q_sol)
    q_prev = q_sol

# ================= NORMAL =================
print("\n===== NORMAL TRAJECTORY =====")

for i, q_goal in enumerate(joint_points):

    print(f"Starting NORMAL Trial {i+1}")
    traj = base_traj(q_A,q_goal)
    acc = run_traj(traj)
    energy = fft_energy(acc)
    normal_results.append(energy)

    print(f"Finished NORMAL Trial {i+1}")
    q_A = q_goal

traj_back = base_traj(q_A, q_start_home)
acc_back = run_traj(traj_back)
return_results.append(fft_energy(acc_back))
print("Finished NORMAL Return to A")

# ================= OPTIMIZED =================
print("\n===== OPTIMIZED TRAJECTORY =====")

q_prev = read_current_angles()

for i, q_goal in enumerate(joint_points):

    print(f"Starting OPTIMIZED Trial {i+1}")

    def objective(c):

        global last_fft,last_acc_cost

        traj = fourier_traj(q_prev,q_goal,c)

        vel = np.gradient(traj, dt, axis=0)
        acc = np.gradient(vel, dt, axis=0)

        J_vel = np.sum(vel**2)
        J_acc = np.sum(acc**2)

        k = np.clip(last_fft/(last_acc_cost+1e-8),0.2,5.0)
        J_fft_pred = k*J_acc

        return J_vel + J_acc + 0.3*J_fft_pred

    result = minimize(objective,np.zeros(DOF),
                      method='Nelder-Mead',
                      options={'maxiter':40})

    traj = fourier_traj(q_prev,q_goal,result.x)
    acc = run_traj(traj)
    energy = fft_energy(acc)

    vel = np.gradient(traj, dt, axis=0)
    acc_calc = np.gradient(vel, dt, axis=0)
    last_acc_cost = np.sum(acc_calc**2)
    last_fft = energy

    opt_results.append(energy)

    print(f"Finished OPTIMIZED Trial {i+1}")
    q_prev = q_goal

traj_back = base_traj(q_prev, q_start_home)
acc_back = run_traj(traj_back)
return_results.append(fft_energy(acc_back))
print("Finished OPTIMIZED Return to A")

# ================= SAVE CSV =================

with open("fft_results.csv","w",newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Trial","Normal_FFT","Optimized_FFT"])

    for i in range(len(normal_results)):
        writer.writerow([i+1,normal_results[i],opt_results[i]])

    writer.writerow([])
    writer.writerow(["Return_to_A_Normal",return_results[0]])
    writer.writerow(["Return_to_A_Optimized",return_results[1]])

print("Results saved.")


===== NORMAL TRAJECTORY =====
Starting NORMAL Trial 1
Finished NORMAL Trial 1
Starting NORMAL Trial 2
Finished NORMAL Trial 2
Starting NORMAL Trial 3
Finished NORMAL Trial 3
Starting NORMAL Trial 4
Finished NORMAL Trial 4
Starting NORMAL Trial 5
Finished NORMAL Trial 5
Finished NORMAL Return to A

===== OPTIMIZED TRAJECTORY =====
Starting OPTIMIZED Trial 1
Finished OPTIMIZED Trial 1
Starting OPTIMIZED Trial 2
Finished OPTIMIZED Trial 2
Starting OPTIMIZED Trial 3
Finished OPTIMIZED Trial 3
Starting OPTIMIZED Trial 4
Finished OPTIMIZED Trial 4
Starting OPTIMIZED Trial 5
Finished OPTIMIZED Trial 5
Finished OPTIMIZED Return to A
Results saved.


## analysing the results

In [1]:
import pandas as pd
import numpy as np

def analyze_fft_results(filename):

    # Read CSV
    df = pd.read_csv(filename)

    # Remove any empty rows (like return rows)
    df = df.dropna()

    normal = df["Normal_FFT"].values
    optimized = df["Optimized_FFT"].values

    print("\n===== PER TRIAL ANALYSIS =====\n")
    print("Trial | Normal | Optimized | % Reduction")
    print("--------------------------------------------")

    reductions = []

    for i in range(len(normal)):

        if normal[i] > 0:
            reduction = 100 * (normal[i] - optimized[i]) / normal[i]
        else:
            reduction = 0

        reductions.append(reduction)

        print(f"{i+1:5d} | "
              f"{normal[i]:7.3f} | "
              f"{optimized[i]:9.3f} | "
              f"{reduction:8.2f}%")

    print("\n===== OVERALL PERFORMANCE =====\n")

    avg_normal = np.mean(normal)
    avg_optimized = np.mean(optimized)

    if avg_normal > 0:
        overall_reduction = 100 * (avg_normal - avg_optimized) / avg_normal
    else:
        overall_reduction = 0

    print(f"Average Normal FFT      : {avg_normal:.4f}")
    print(f"Average Optimized FFT   : {avg_optimized:.4f}")
    print(f"Overall FFT Reduction   : {overall_reduction:.2f}%")

    print(f"\nStd Dev Normal          : {np.std(normal):.4f}")
    print(f"Std Dev Optimized       : {np.std(optimized):.4f}")
    print(f"Mean % Reduction        : {np.mean(reductions):.2f}%")

    if overall_reduction > 0:
        print("\nOptimization successfully reduced vibration ✔")
    else:
        print("\nNo overall improvement")


# Run analysis
analyze_fft_results("fft_results.csv")


===== PER TRIAL ANALYSIS =====

Trial | Normal | Optimized | % Reduction
--------------------------------------------
    1 |  26.553 |     3.223 |    87.86%
    2 |   6.602 |     0.952 |    85.58%
    3 |   9.835 |     2.208 |    77.55%
    4 |  42.485 |    35.647 |    16.10%
    5 |  20.048 |    10.133 |    49.46%
    6 |   1.440 |     0.370 |    74.34%

===== OVERALL PERFORMANCE =====

Average Normal FFT      : 17.8272
Average Optimized FFT   : 8.7554
Overall FFT Reduction   : 50.89%

Std Dev Normal          : 13.8293
Std Dev Optimized       : 12.4481
Mean % Reduction        : 65.15%

Optimization successfully reduced vibration ✔


## single optimised demo

In [11]:
import numpy as np
import serial
import time
import csv
from scipy.fft import fft, fftfreq
from scipy.optimize import minimize

# =====================================================
# SERIAL
# =====================================================
ser = serial.Serial('COM4',115200,timeout=1)
time.sleep(2)

# =====================================================
# ROBOT PARAMETERS
# =====================================================
DOF = 4
T = 80
dt = 0.04

L1 = 0.05
L2 = 0.11
L3 = 0.11

# =====================================================
# PREDICTIVE MODEL STORAGE
# =====================================================
last_fft = 1.0
last_acc_cost = 1.0

# =====================================================
# FORWARD KINEMATICS
# =====================================================
def forward_kinematics(q):

    q1,q2,q3,q4 = q

    r = L2*np.cos(q2) + L3*np.cos(q2+q3)
    x = np.cos(q1)*r
    y = np.sin(q1)*r
    z = L1 + L2*np.sin(q2) + L3*np.sin(q2+q3)

    return np.array([x,y,z])

# =====================================================
# READ CURRENT ANGLES
# =====================================================
def read_current_angles():

    while True:
        line = ser.readline().decode(errors='ignore').strip()

        if line.startswith("ANG:"):
            try:
                angles = line.replace("ANG:","")
                base, shoulder, elbow, gripper = map(float, angles.split(","))
                return np.radians([base, shoulder, elbow, gripper])
            except:
                continue

# =====================================================
# JACOBIAN
# =====================================================
def jacobian(q):

    q1,q2,q3,_ = q
    r = L2*np.cos(q2) + L3*np.cos(q2+q3)

    J = np.zeros((3,3))

    J[0,0] = -np.sin(q1)*r
    J[1,0] =  np.cos(q1)*r

    J[0,1] = np.cos(q1)*(-L2*np.sin(q2)-L3*np.sin(q2+q3))
    J[1,1] = np.sin(q1)*(-L2*np.sin(q2)-L3*np.sin(q2+q3))
    J[2,1] = L2*np.cos(q2)+L3*np.cos(q2+q3)

    J[0,2] = np.cos(q1)*(-L3*np.sin(q2+q3))
    J[1,2] = np.sin(q1)*(-L3*np.sin(q2+q3))
    J[2,2] = L3*np.cos(q2+q3)

    return J

# =====================================================
# IK
# =====================================================
def inverse_kinematics_NR(target,q_init):

    q = q_init.copy()

    for _ in range(50):

        error = target - forward_kinematics(q)

        if np.linalg.norm(error) < 1e-4:
            break

        dq = np.linalg.pinv(jacobian(q)) @ error
        q[:3] += dq

    q[3] = 0
    return q

# =====================================================
# TRAJECTORIES
# =====================================================
def base_traj(q0,qf):
    t = np.linspace(0,1,T)
    return q0 + np.outer(t,(qf-q0))

def fourier_traj(q0,qf,c):
    t = np.linspace(0,1,T)
    q = base_traj(q0,qf)
    for i in range(DOF):
        q[:,i] += 0.25*c[i]*np.sin(2*np.pi*3*t)
    return q

# =====================================================
# SERVO EXECUTION
# =====================================================
def clamp_servo(rad):

    deg = np.degrees(rad)

    deg[0] = np.clip(deg[0],0,180)
    deg[1] = np.clip(deg[1],0,180)
    deg[2] = np.clip(deg[2],90,180)
    deg[3] = np.clip(deg[3],0,180)

    return deg

def run_traj(traj):

    acc_data = []

    for i in range(T):

        deg = clamp_servo(traj[i])
        deg[3] = 100   # keep gripper open

        packet = " ".join(str(int(a)) for a in deg) + "\n"
        ser.write(packet.encode())

        line = ser.readline().decode(errors='ignore').strip()

        if line:
            try:
                ax,ay,az = map(float,line.split(","))
                acc_data.append([ax,ay,az])
            except:
                pass

        time.sleep(dt)

    deg = clamp_servo(traj[-1])

    deg[3] = 100   # close
    ser.write((" ".join(str(int(a)) for a in deg) + "\n").encode())
    time.sleep(1)

    deg[3] = 0 # hold
    ser.write((" ".join(str(int(a)) for a in deg) + "\n").encode())

    return np.array(acc_data)

# =====================================================
# FFT ENERGY
# =====================================================
def fft_energy(acc):

    if len(acc)<10:
        return 1e9

    acc -= np.mean(acc,axis=0)
    N = len(acc)
    freqs = fftfreq(N,dt)

    fft_total = (
        np.abs(fft(acc[:,0]))**2 +
        np.abs(fft(acc[:,1]))**2 +
        np.abs(fft(acc[:,2]))**2
    )/N

    pos = freqs>0
    freqs = freqs[pos]
    fft_total = fft_total[pos]

    band = (freqs>=4)&(freqs<=7)

    return np.sum(fft_total[band])


# ================= SINGLE OPTIMIZED DEMO =================

print("\n===== DEMO: Optimized A -> F =====")

# Read start position
q_start = read_current_angles()

# Define F
pos_F = np.array([0.23, 0.0, 0.12])   # strong negative base rotation

# Solve IK
q_goal = inverse_kinematics_NR(pos_F, q_start)

print("Optimizing trajectory...")


def objective(c):

    global last_fft,last_acc_cost

    traj = fourier_traj(q_start,q_goal,c)

    vel = np.gradient(traj, dt, axis=0)
    acc = np.gradient(vel, dt, axis=0)

    J_vel = np.sum(vel**2)
    J_acc = np.sum(acc**2)

    k = np.clip(last_fft/(last_acc_cost+1e-8),0.2,5.0)
    J_fft_pred = k*J_acc

    return J_vel + J_acc + 0.3*J_fft_pred

result = minimize(objective,
                  np.zeros(DOF),
                  method='Nelder-Mead',
                  options={'maxiter':40})

print("Executing optimized trajectory...")
traj_opt = fourier_traj(q_start, q_goal, result.x)
acc_data = run_traj(traj_opt)

energy = fft_energy(acc_data)

print("Optimized FFT Energy:", energy)


# ================= MOVE F -> B =================

print("\nMoving from F to B (Elbow rotation demo)...")

pos_B = np.array([0.23, 0.0, 0.05])

q_B = inverse_kinematics_NR(pos_B, q_goal)

traj_FB = base_traj(q_goal, q_B)

acc_FB = run_traj(traj_FB)

energy_FB = fft_energy(acc_FB)

print("FFT Energy F -> B:", energy_FB)

print("Demo complete.")


===== DEMO: Optimized A -> F =====
Optimizing trajectory...
Executing optimized trajectory...
Optimized FFT Energy: 14.230824740112853

Moving from F to B (Elbow rotation demo)...
FFT Energy F -> B: 247.92751430608433
Demo complete.
